# Yelp Restaurant Data Cleaning
**Team:** Maya, Tabesh, Alexa
**Goal of this notebook:** Take the raw Yelp business file and turn it into one clean table of restaurants that we can use for charts and machine learning.

**What goes in:** `yelp_academic_dataset_business.json` (all ~150,000 Yelp businesses)
**What comes out:** `restaurants_clean.csv` (one row per restaurant, with neat columns)

### The steps in this notebook
1. Setup and load the data
2. Take a first look
3. Keep only restaurants
4. Fix locations (metro area, city names, ZIP codes)
5. Unpack the `attributes` column (price, noise, alcohol, etc.)
6. Unpack the nested attributes (ambience, parking, meals)
7. Turn `categories` into a cuisine column
8. Turn `hours` into simple features
9. Make the target columns
10. Final checks (duplicates, missing values)
11. Save the clean file

> **AI note:** This notebook was first written by Claude (an AI assistant) and then reviewed and run by our team. See the AI documentation section at the bottom.

## Step 1: Setup and load the data

**What this does:** Loads the Python tools we need and reads the Yelp business file into a table (called a *DataFrame*).

**Why `lines=True`?** Yelp puts one business on each line of the file instead of one big list, so we have to tell pandas to read it line by line.

**Why `dtype={"postal_code": str}`?** Without it, pandas treats ZIP codes as numbers and removes leading zeros. For example, New Jersey ZIP `08002` would turn into `8002`, which would break our Census matching later.

👉 **If you get a "file not found" error**, change `DATA_PATH` below so it points to where your file is.

In [2]:
# Run this once if pandas is not installed yet (remove the # to run it):
# %pip install pandas numpy

import ast            # helps turn text that looks like a Python dictionary into a real dictionary
import numpy as np    # for math and for marking missing values (np.nan)
import pandas as pd   # for working with tables

# Show all columns when we print a table (instead of hiding some with "...")
pd.set_option("display.max_columns", 100)

# Where the raw Yelp file lives (change this if your folders are different)
DATA_PATH = "/Users/tabeshakbari/Desktop/machine learning project/data/yelp_cleaned_dataset/yelp_academic_dataset_business.json"

# Where we will save the clean file at the end
OUTPUT_PATH = "/Users/tabeshakbari/Desktop/machine learning project/data/restaurants_clean_v2.csv"

# Read the file: one business per line, and keep ZIP codes as text
business = pd.read_json(DATA_PATH, lines=True, dtype={"postal_code": str})

print("Rows (businesses):", business.shape[0])
print("Columns:", business.shape[1])

Rows (businesses): 150346
Columns: 14


## Step 2: Take a first look

**What this does:** Shows the first few rows, the column names and types, and how many values are missing in each column.

**What to notice:**
- `attributes` and `hours` are *dictionaries* stuck inside a single column. We need to unpack them (Steps 5, 6 and 8).
- `categories` is one long piece of text like `"Pizza, Italian, Restaurants"`.

In [3]:
# First 3 rows
business.head(3)

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."


In [4]:
# Column names, data types, and how many non-missing values each column has
business.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150346 entries, 0 to 150345
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   business_id   150346 non-null  object 
 1   name          150346 non-null  object 
 2   address       150346 non-null  object 
 3   city          150346 non-null  object 
 4   state         150346 non-null  object 
 5   postal_code   150346 non-null  object 
 6   latitude      150346 non-null  float64
 7   longitude     150346 non-null  float64
 8   stars         150346 non-null  float64
 9   review_count  150346 non-null  int64  
 10  is_open       150346 non-null  int64  
 11  attributes    136602 non-null  object 
 12  categories    150243 non-null  object 
 13  hours         127123 non-null  object 
dtypes: float64(3), int64(2), object(9)
memory usage: 16.1+ MB


In [5]:
# Look at what ONE restaurant's attributes look like (it's a dictionary)
business.loc[business["categories"].str.contains("Restaurants", na=False), "attributes"].iloc[0]

{'RestaurantsDelivery': 'False',
 'OutdoorSeating': 'False',
 'BusinessAcceptsCreditCards': 'False',
 'BusinessParking': "{'garage': False, 'street': True, 'validated': False, 'lot': False, 'valet': False}",
 'BikeParking': 'True',
 'RestaurantsPriceRange2': '1',
 'RestaurantsTakeOut': 'True',
 'ByAppointmentOnly': 'False',
 'WiFi': "u'free'",
 'Alcohol': "u'none'",
 'Caters': 'True'}

## Step 3: Keep only restaurants

**What this does:** The file has every kind of business (dentists, salons, car shops...). We keep only rows where the `categories` text includes the word **"Restaurants"**.

`na=False` means: if a business has no categories at all, treat it as *not* a restaurant.

`.copy()` makes a separate table so pandas doesn't give us warnings when we change it later.

In [6]:
is_restaurant = business["categories"].str.contains("Restaurants", na=False)
restaurants = business[is_restaurant].copy()

print("All businesses:", len(business))
print("Restaurants only:", len(restaurants))

All businesses: 150346
Restaurants only: 52268


## Step 4: Fix locations

### 4a. Add a `metro` column
**What this does:** Yelp covers 11 metro areas, but the `city` column has hundreds of names because it includes suburbs. Each metro is in its own state, so we use **state** to label the metro area.

A few metros cross state lines:
- **Philadelphia** includes suburbs in **NJ** (like Cherry Hill) and **DE** (like Wilmington)
- **St. Louis** includes suburbs in **IL**

There are also a handful of stray rows in other states (like CO or HI) that don't belong to any of the 11 metros. We remove those.

In [7]:
# Which states show up, and how many restaurants each has
restaurants["state"].value_counts()

state
PA     12641
FL      8731
TN      4352
MO      4247
IN      4150
LA      3640
NJ      3341
AZ      2671
AB      2410
NV      1673
ID      1302
CA      1161
IL       983
DE       961
NC         1
CO         1
HI         1
MT         1
XMS        1
Name: count, dtype: int64

In [8]:
# Map each state to its metro area
state_to_metro = {
    "PA": "Philadelphia", "NJ": "Philadelphia", "DE": "Philadelphia",
    "FL": "Tampa",
    "IN": "Indianapolis",
    "TN": "Nashville",
    "AZ": "Tucson",
    "LA": "New Orleans",
    "MO": "St. Louis", "IL": "St. Louis",
    "NV": "Reno",
    "ID": "Boise",
    "CA": "Santa Barbara",
    "AB": "Edmonton",
}

restaurants["metro"] = restaurants["state"].map(state_to_metro)

# Rows whose state isn't in our list get NaN (missing). Drop them.
before = len(restaurants)
restaurants = restaurants[restaurants["metro"].notna()].copy()
print("Removed stray rows:", before - len(restaurants))

restaurants["metro"].value_counts()

Removed stray rows: 5


metro
Philadelphia     16943
Tampa             8731
St. Louis         5230
Nashville         4352
Indianapolis      4150
New Orleans       3640
Tucson            2671
Edmonton          2410
Reno              1673
Boise             1302
Santa Barbara     1161
Name: count, dtype: int64

### 4b. Flag U.S. vs. Canada
**What this does:** Edmonton is in Canada, so U.S. Census data won't cover it. Instead of deleting Edmonton, we add a yes/no column `in_us`. Later we can leave Edmonton out of the Census questions but keep it for the others.

In [9]:
restaurants["in_us"] = (restaurants["state"] != "AB").astype(int)   # 1 = U.S., 0 = Canada
restaurants["in_us"].value_counts()

in_us
1    49853
0     2410
Name: count, dtype: int64

### 4c. Clean up city names
**What this does:** The same city is spelled different ways, like `"St. Louis"` vs `"Saint Louis"` or `"St. Petersburg"` vs `"Saint Petersburg"`. We:
1. Remove extra spaces at the start/end
2. Make capitalization consistent (Title Case)
3. Change a starting `"St."` or `"St "` to `"Saint "`
4. Fix a few common abbreviations

This isn't perfect, but it fixes the biggest problems.

In [10]:
print("Different city names BEFORE:", restaurants["city"].nunique())

city = restaurants["city"].str.strip().str.title()
city = city.str.replace(r"^St\.?\s+", "Saint ", regex=True)          # "St. Louis" -> "Saint Louis"
city = city.str.replace(r"\s+", " ", regex=True)                    # remove double spaces
city = city.replace({"Phila": "Philadelphia", "Philly": "Philadelphia"})
restaurants["city"] = city

print("Different city names AFTER:", restaurants["city"].nunique())
restaurants["city"].value_counts().head(15)

Different city names BEFORE: 917
Different city names AFTER: 825


city
Philadelphia        5859
Tampa               2967
Indianapolis        2863
Nashville           2505
Saint Louis         2484
Tucson              2469
New Orleans         2261
Edmonton            2168
Reno                1288
Saint Petersburg    1017
Boise                848
Santa Barbara        768
Clearwater           679
Wilmington           621
Metairie             522
Name: count, dtype: int64

### 4d. Clean ZIP codes
**What this does:** For U.S. restaurants, we keep only the first 5 characters of the ZIP code (some are written like `12345-6789`). If a U.S. ZIP code isn't exactly 5 digits, we mark it as missing. Canadian postal codes (like `T5J 1B9`) are left as they are.

We need clean ZIP codes to join with Census data later.

In [11]:
zip5 = restaurants["postal_code"].str.strip().str[:5]
looks_valid = zip5.str.fullmatch(r"\d{5}", na=False)

# Only touch U.S. rows; keep Canada as-is
us = restaurants["in_us"] == 1
restaurants.loc[us, "postal_code"] = zip5[us].where(looks_valid[us], np.nan)

print("U.S. restaurants with a missing/bad ZIP:", restaurants.loc[us, "postal_code"].isna().sum())

U.S. restaurants with a missing/bad ZIP: 19


## Step 5: Unpack the `attributes` column

**The problem:** Each restaurant's `attributes` is a dictionary, and the values are messy text, for example:
- `"True"` / `"False"` (text, not real yes/no values)
- `"u'average'"` or `"'average'"` (extra `u` and quote marks from old Python formatting)
- `"None"` (means Yelp doesn't know)

**The fix:** We write a small helper function, `clean_value`, that:
- removes the `u'...'` and quote marks
- turns `"True"` into `1` and `"False"` into `0`
- turns `"None"` into a real missing value (`NaN`)

Then we pull out each attribute we care about into its own column with an easy name.

⚠️ **Careful:** For `Alcohol`, the value `'none'` (lowercase) is a real answer meaning *no alcohol served*. Only `None` (capital N, no quotes) means *missing*. The function handles this correctly.

In [12]:
def clean_value(value):
    """Turn one messy Yelp attribute value into a clean value."""
    # Missing in the data -> NaN
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan

    text = str(value).strip()

    # "None" (no quotes) means Yelp doesn't know
    if text == "None":
        return np.nan

    # Remove old-style Python text markers: u'average' -> average, 'average' -> average
    if text.startswith("u'") or text.startswith('u"'):
        text = text[1:]
    text = text.strip("'\"")

    # Yes/no values -> 1/0
    if text == "True":
        return 1
    if text == "False":
        return 0

    return text


# Quick test so we can see it works
for example in ["True", "False", "None", "u'average'", "'full_bar'", "u'none'", "2"]:
    print(f"{example!r:15} -> {clean_value(example)!r}")

'True'          -> 1
'False'         -> 0
'None'          -> nan
"u'average'"    -> 'average'
"'full_bar'"    -> 'full_bar'
"u'none'"       -> 'none'
'2'             -> '2'


In [13]:
def get_attribute(attr_dict, key):
    """Safely get one attribute from a restaurant's attributes dictionary."""
    if not isinstance(attr_dict, dict):      # some restaurants have no attributes at all
        return np.nan
    return clean_value(attr_dict.get(key))   # .get() returns None if the key is missing


# Yelp's attribute name  ->  our easier column name
attribute_names = {
    # Main features from Research Question 1
    "RestaurantsPriceRange2":   "price_level",        # 1 = $, 2 = $$, 3 = $$$, 4 = $$$$
    "NoiseLevel":               "noise_level",        # quiet / average / loud / very_loud
    "OutdoorSeating":           "outdoor_seating",
    "RestaurantsDelivery":      "delivery",
    "RestaurantsTakeOut":       "takeout",
    "Alcohol":                  "alcohol",            # none / beer_and_wine / full_bar
    # Extra features that might help the model
    "RestaurantsReservations":  "reservations",
    "RestaurantsGoodForGroups": "good_for_groups",
    "GoodForKids":              "good_for_kids",
    "HasTV":                    "has_tv",
    "WiFi":                     "wifi",               # no / free / paid
    "RestaurantsAttire":        "attire",             # casual / dressy / formal
    "BikeParking":              "bike_parking",
    "Caters":                   "caters",
    "RestaurantsTableService":  "table_service",
    "WheelchairAccessible":     "wheelchair_accessible",
    "HappyHour":                "happy_hour",
    "DogsAllowed":              "dogs_allowed",
    "BusinessAcceptsCreditCards": "accepts_credit_cards",
    "DriveThru":                "drive_thru",
}

# Make one new column for each attribute
for yelp_name, new_name in attribute_names.items():
    restaurants[new_name] = restaurants["attributes"].apply(lambda d: get_attribute(d, yelp_name))

restaurants[list(attribute_names.values())].head()

,price_level,noise_level,outdoor_seating,delivery,takeout,alcohol,reservations,good_for_groups,good_for_kids,has_tv,wifi,attire,bike_parking,caters,table_service,wheelchair_accessible,happy_hour,dogs_allowed,accepts_credit_cards,drive_thru
3,1,NaN,0.0,0.0,1.0,none,NaN,NaN,NaN,NaN,free,NaN,1.0,1.0,NaN,NaN,NaN,NaN,0.0,NaN
5,1,NaN,1.0,1.0,1.0,none,0.0,1.0,1.0,1.0,no,casual,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0
8,1,average,1.0,0.0,1.0,full_bar,0.0,1.0,1.0,1.0,free,casual,NaN,1.0,NaN,NaN,NaN,NaN,1.0,NaN
9,1,NaN,1.0,1.0,1.0,none,0.0,0.0,1.0,1.0,no,casual,NaN,0.0,0.0,1.0,0.0,0.0,1.0,1.0
11,NaN,NaN,NaN,NaN,NaN,none,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 5b. Fix data types
**What this does:**
- `price_level` is text like `"2"`, so we turn it into a number.
- `noise_level` has a natural order (quiet < average < loud < very_loud), so we also make a number version: `noise_level_num` (1 to 4). Models can use the order this way.
- Yes/no columns become numbers (1, 0, or missing).

In [14]:
# Price: text -> number
restaurants["price_level"] = pd.to_numeric(restaurants["price_level"], errors="coerce")

# Noise: make an ordered number version
noise_order = {"quiet": 1, "average": 2, "loud": 3, "very_loud": 4}
restaurants["noise_level_num"] = restaurants["noise_level"].map(noise_order)

# Yes/no columns: make sure they are numbers
yes_no_cols = [
    "outdoor_seating", "delivery", "takeout", "reservations", "good_for_groups",
    "good_for_kids", "has_tv", "bike_parking", "caters", "table_service",
    "wheelchair_accessible", "happy_hour", "dogs_allowed", "accepts_credit_cards", "drive_thru",
]
for col in yes_no_cols:
    restaurants[col] = pd.to_numeric(restaurants[col], errors="coerce")

# Check the text columns now have clean values
for col in ["noise_level", "alcohol", "wifi", "attire"]:
    print(col, "->", restaurants[col].value_counts(dropna=False).to_dict())
print("price_level ->", restaurants["price_level"].value_counts(dropna=False).sort_index().to_dict())

noise_level -> {'average': 24443, nan: 17457, 'quiet': 7162, 'loud': 2364, 'very_loud': 837}
alcohol -> {'none': 20301, 'full_bar': 13894, nan: 12320, 'beer_and_wine': 5748}
wifi -> {'free': 20420, 'no': 17028, nan: 14554, 'paid': 261}
attire -> {'casual': 38086, nan: 13311, 'dressy': 796, 'formal': 70}
price_level -> {1.0: 19508, 2.0: 23117, 3.0: 1662, 4.0: 194, nan: 7782}


## Step 6: Unpack the nested attributes (ambience, parking, meals)

**The problem:** Some attributes are a dictionary *inside* the dictionary, stored as text. For example, `Ambience` looks like:

`"{'romantic': False, 'casual': True, 'trendy': False, ...}"`

**The fix:**
1. Use `ast.literal_eval` to safely turn that text into a real dictionary.
2. Make one yes/no column for each item, with a prefix so we know where it came from:
   - `ambience_casual`, `ambience_romantic`, ...
   - `parking_lot`, `parking_street`, ...
   - `meal_lunch`, `meal_dinner`, ...
3. Add `parking_any` = 1 if the restaurant has *any* kind of parking.

In [15]:
def text_to_dict(text):
    """Turn text like "{'casual': True}" into a real dictionary. Return {} if it can't."""
    if not isinstance(text, str) or text.strip() in ("None", ""):
        return {}
    try:
        result = ast.literal_eval(text)   # safe: only reads simple Python values, never runs code
        return result if isinstance(result, dict) else {}
    except (ValueError, SyntaxError):
        return {}


def unpack_nested(df, yelp_name, prefix, keys):
    """Make one 1/0 column per key inside a nested attribute like Ambience."""
    inner = df["attributes"].apply(
        lambda d: text_to_dict(d.get(yelp_name)) if isinstance(d, dict) else {}
    )
    for key in keys:
        # True -> 1, False -> 0, not listed / None -> NaN
        df[f"{prefix}_{key}"] = inner.apply(
            lambda x: np.nan if x.get(key) is None else int(bool(x.get(key)))
        )
    return df


ambience_keys = ["casual", "classy", "divey", "hipster", "intimate", "romantic", "touristy", "trendy", "upscale"]
parking_keys  = ["garage", "street", "lot", "valet", "validated"]
meal_keys     = ["breakfast", "brunch", "lunch", "dinner", "dessert", "latenight"]

restaurants = unpack_nested(restaurants, "Ambience",        "ambience", ambience_keys)
restaurants = unpack_nested(restaurants, "BusinessParking", "parking",  parking_keys)
restaurants = unpack_nested(restaurants, "GoodForMeal",     "meal",     meal_keys)

# parking_any: 1 if any parking type is 1, 0 if all known types are 0, NaN if we know nothing
parking_cols = [f"parking_{k}" for k in parking_keys]
restaurants["parking_any"] = restaurants[parking_cols].max(axis=1)

restaurants[[f"ambience_{k}" for k in ambience_keys] + ["parking_any"]].head()

,ambience_casual,ambience_classy,ambience_divey,ambience_hipster,ambience_intimate,ambience_romantic,ambience_touristy,ambience_trendy,ambience_upscale,parking_any
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Step 7: Turn `categories` into a cuisine column

**The problem:** `categories` is text like `"Pizza, Italian, Restaurants, Food"`. Restaurants have many categories, and some are not cuisines at all ("Restaurants", "Food").

**The fix:**
1. Split the text into a list of categories.
2. Use a lookup table that groups Yelp categories into about 20 bigger cuisine groups (for example, "Sushi Bars" and "Ramen" both go into **Japanese**).
3. If a restaurant matches more than one group, we pick the one that appears **first in our priority list**. Specific cuisines (Mexican, Thai...) come before general types (Fast Food, Bars), because they tell us more. For example, `"Pizza, Italian"` becomes **Italian**.
4. Anything we can't match becomes **Other**.

We also make a few extra simple columns:
- `num_categories`: how many categories the restaurant lists
- `is_fast_food`, `is_bar`: yes/no flags
- `is_chain`: 1 if 5 or more restaurants in the data share the same name (like McDonald's)

In [16]:
# Turn the text into a clean list: "Pizza, Italian" -> ["Pizza", "Italian"]
restaurants["category_list"] = restaurants["categories"].str.split(",").apply(
    lambda cats: [c.strip() for c in cats]
)

# Cuisine groups in PRIORITY order (first match wins).
# Left side = our group name, right side = Yelp categories that belong to it.
cuisine_groups = {
    "Mexican":            ["Mexican", "Tex-Mex", "Tacos", "New Mexican Cuisine"],
    "Italian":            ["Italian"],
    "Chinese":            ["Chinese", "Dim Sum", "Cantonese", "Szechuan", "Taiwanese", "Hot Pot"],
    "Japanese":           ["Japanese", "Sushi Bars", "Ramen", "Izakaya"],
    "Thai":               ["Thai"],
    "Vietnamese":         ["Vietnamese"],
    "Indian":             ["Indian", "Pakistani", "Himalayan/Nepalese"],
    "Korean":             ["Korean"],
    "French":             ["French"],
    "Vegetarian & Vegan": ["Vegan", "Vegetarian"],
    "Mediterranean & Middle Eastern": ["Mediterranean", "Middle Eastern", "Greek", "Lebanese",
                                       "Turkish", "Falafel", "Halal", "Persian/Iranian"],
    "Latin & Caribbean":  ["Latin American", "Caribbean", "Cuban", "Puerto Rican", "Peruvian",
                           "Colombian", "Salvadoran", "Venezuelan", "Dominican", "Brazilian"],
    "Southern & Cajun":   ["Southern", "Cajun/Creole", "Soul Food"],
    "Other Asian":        ["Asian Fusion", "Pan Asian", "Filipino", "Hawaiian", "Poke", "Malaysian"],
    "Seafood":            ["Seafood", "Fish & Chips"],
    "Steakhouse":         ["Steakhouses"],
    "Barbeque":           ["Barbeque"],
    "Pizza":              ["Pizza"],
    "Burgers":            ["Burgers"],
    "Sandwiches & Delis": ["Sandwiches", "Delis", "Cheesesteaks"],
    "Breakfast & Brunch": ["Breakfast & Brunch", "Bagels", "Pancakes", "Waffles"],
    "Cafe & Bakery":      ["Cafes", "Coffee & Tea", "Bakeries", "Desserts", "Juice Bars & Smoothies",
                           "Ice Cream & Frozen Yogurt", "Donuts"],
    "Fast Food":          ["Fast Food", "Chicken Wings", "Hot Dogs", "Chicken Shop"],
    "American":           ["American (Traditional)", "American (New)", "Diners", "Comfort Food"],
    "Bar & Pub":          ["Bars", "Pubs", "Sports Bars", "Cocktail Bars", "Wine Bars", "Beer Bar",
                           "Lounges", "Gastropubs", "Breweries"],
}


def pick_cuisine(cat_list):
    """Return the first cuisine group (in priority order) that matches any category."""
    cats = set(cat_list)
    for group, yelp_cats in cuisine_groups.items():   # dictionaries keep their order
        if cats.intersection(yelp_cats):
            return group
    return "Other"


restaurants["cuisine"] = restaurants["category_list"].apply(pick_cuisine)

# Extra simple features
restaurants["num_categories"] = restaurants["category_list"].apply(len)
restaurants["is_fast_food"] = restaurants["category_list"].apply(lambda c: int("Fast Food" in c))
restaurants["is_bar"] = restaurants["category_list"].apply(
    lambda c: int(bool(set(c) & {"Bars", "Pubs", "Sports Bars", "Cocktail Bars", "Wine Bars", "Beer Bar"}))
)

# Chain: same name appears 5+ times (compare names in lowercase so "Subway" = "SUBWAY")
name_counts = restaurants["name"].str.strip().str.lower().map(
    restaurants["name"].str.strip().str.lower().value_counts()
)
restaurants["is_chain"] = (name_counts >= 5).astype(int)

restaurants["cuisine"].value_counts()

cuisine
Sandwiches & Delis                5335
Mexican                           4768
Italian                           4528
Burgers                           4276
Pizza                             4067
American                          3745
Chinese                           3211
Breakfast & Brunch                2593
Cafe & Bakery                     2365
Japanese                          1975
Seafood                           1860
Southern & Cajun                  1822
Mediterranean & Middle Eastern    1563
Fast Food                         1327
Vegetarian & Vegan                1135
Latin & Caribbean                 1131
Other                             1014
Barbeque                           975
Indian                             810
Steakhouse                         771
Bar & Pub                          628
Vietnamese                         611
Thai                               589
Other Asian                        455
French                             428
Korean           

## Step 8: Turn `hours` into simple features

**The problem:** `hours` is a dictionary like `{"Monday": "7:0-20:0", "Friday": "11:0-2:0", ...}`. The times use a 24-hour clock, and `"11:0-2:0"` means it closes at 2 AM the next morning.

**The fix:** We make three easy columns:
- `days_open`: how many days per week it's open (0–7)
- `open_weekend`: 1 if it's open on Saturday or Sunday
- `open_late`: 1 if on any day it closes at 10 PM or later (including after midnight)

Restaurants with no hours listed get missing values (`NaN`).

Note: Yelp sometimes writes `"0:0-0:0"`. That can mean "open 24 hours" or "unknown", so we count the day as open but don't count it as "open late".

In [17]:
def to_hours(time_text):
    """'17:30' -> 17.5"""
    h, m = time_text.split(":")
    return int(h) + int(m) / 60


def hours_features(hours_dict):
    """Return days_open, open_weekend, open_late for one restaurant."""
    if not isinstance(hours_dict, dict) or len(hours_dict) == 0:
        return pd.Series({"days_open": np.nan, "open_weekend": np.nan, "open_late": np.nan})

    open_late = 0
    for day, span in hours_dict.items():
        try:
            start, end = span.split("-")
            start, end = to_hours(start), to_hours(end)
        except ValueError:
            continue                      # skip anything oddly formatted
        if start == 0 and end == 0:
            continue                      # "0:0-0:0" is unclear, so don't count as late
        if end >= 22 or end < start:      # closes 10pm+ or after midnight
            open_late = 1

    return pd.Series({
        "days_open": len(hours_dict),
        "open_weekend": int("Saturday" in hours_dict or "Sunday" in hours_dict),
        "open_late": open_late,
    })


restaurants[["days_open", "open_weekend", "open_late"]] = restaurants["hours"].apply(hours_features)

restaurants[["days_open", "open_weekend", "open_late"]].describe()

,days_open,open_weekend,open_late
count,44988.000000,44988.000000,44988.000000
mean,6.552503,0.972126,0.557326
std,0.848507,0.164614,0.496708
min,1.000000,0.000000,0.000000
25%,6.000000,1.000000,0.000000
50%,7.000000,1.000000,1.000000
75%,7.000000,1.000000,1.000000
max,7.000000,1.000000,1.000000


## Step 9: Make the target columns

**What this does:** Creates the columns our models will try to predict.

- **`high_rating`** (main target): 1 if `stars` is **4.0 or higher**, otherwise 0
- **`stars`** (alternative target): keep as a number for a regression model
- **`is_open`** (second success measure, Research Question 5): already 1 = open, 0 = closed

⚠️ **Data leakage reminder:** When we predict `high_rating`, we must **not** use `stars` as an input feature, because `high_rating` is made directly from `stars`. That would be like giving the model the answer.

In [18]:
restaurants["high_rating"] = (restaurants["stars"] >= 4).astype(int)

print("Share of restaurants with 4+ stars:", round(restaurants["high_rating"].mean(), 3))
print("Share still open:", round(restaurants["is_open"].mean(), 3))
print()
print("Share with 4+ stars by metro:")
restaurants.groupby("metro")["high_rating"].mean().sort_values(ascending=False).round(3)

Share of restaurants with 4+ stars: 0.447
Share still open: 0.669

Share with 4+ stars by metro:


metro
Santa Barbara    0.579
New Orleans      0.508
Tampa            0.480
Reno             0.461
Nashville        0.451
Boise            0.451
Indianapolis     0.442
Philadelphia     0.431
Edmonton         0.422
St. Louis        0.401
Tucson           0.395
Name: high_rating, dtype: float64

## Step 10: Final checks

### 10a. Duplicates and impossible values
**What this does:** Makes sure every restaurant appears only once (`business_id` should be unique) and that stars and map coordinates are in sensible ranges.

In [20]:
print("Duplicate business_id rows:", restaurants["business_id"].duplicated().sum())
print("Stars range:", restaurants["stars"].min(), "to", restaurants["stars"].max())
print("Latitude range:", round(restaurants["latitude"].min(), 2), "to", round(restaurants["latitude"].max(), 2))
print("Longitude range:", round(restaurants["longitude"].min(), 2), "to", round(restaurants["longitude"].max(), 2))

Duplicate business_id rows: 0
Stars range: 1.0 to 5.0
Latitude range: 27.56 to 53.68
Longitude range: -120.08 to -74.66


### 10b. Remove columns we don't need anymore
**What this does:** We already pulled everything useful out of `attributes`, `hours`, and `category_list`, so we drop them. We **keep** `categories` (the original text) in case we want it later, and `name` + `address` because we'll need them to match health inspection records.

In [21]:
restaurants = restaurants.drop(columns=["attributes", "hours", "category_list"])
print("Columns now:", restaurants.shape[1])

Columns now: 65


### 10c. Missing values
**What this does:** Shows what percent of each column is missing.

**Why we are NOT filling in missing values yet:** Many restaurants simply didn't list things like noise level or ambience. How we handle that (fill with the most common value, add a "missing" category, or drop the column) is a modeling choice. It should be done **after** we split into training and test sets in Week 4, so the test data doesn't "leak" into how we fill values. For now, we just record it.

Things to discuss as a team:
- Columns missing for most restaurants (like `drive_thru` or `dogs_allowed`) might not be worth using.
- "Missing" itself might mean something. For example, a restaurant that lists very few attributes may be less established.

In [22]:
missing_pct = (restaurants.isna().mean() * 100).round(1).sort_values(ascending=False)
missing_pct[missing_pct > 0]

drive_thru               88.8
dogs_allowed             78.1
happy_hour               75.1
wheelchair_accessible    74.1
table_service            62.3
meal_dessert             54.2
meal_latenight           52.5
meal_brunch              52.3
meal_breakfast           50.1
meal_lunch               50.0
meal_dinner              48.0
caters                   33.9
noise_level_num          33.4
noise_level              33.4
bike_parking             32.2
ambience_divey           31.3
ambience_trendy          31.0
ambience_hipster         29.3
ambience_intimate        29.0
ambience_classy          27.9
ambience_touristy        27.9
wifi                     27.8
ambience_romantic        27.4
ambience_upscale         26.6
attire                   25.5
ambience_casual          25.2
alcohol                  23.6
good_for_kids            21.6
good_for_groups          20.8
outdoor_seating          19.9
has_tv                   19.8
parking_street           18.9
reservations             18.0
parking_lo

## Step 11: Save the clean file

**What this does:** Saves our clean restaurant table as a CSV file. Teammates can load this smaller file instead of re-running all the cleaning.

To load it later:
```python
restaurants = pd.read_csv("data/restaurants_clean.csv", dtype={"postal_code": str})
```
(Again, `dtype={"postal_code": str}` keeps leading zeros in ZIP codes.)

In [23]:
# Put the columns in a nice order: IDs and location first, targets next, then features
first_cols = ["business_id", "name", "address", "city", "state", "postal_code", "metro", "in_us",
              "latitude", "longitude", "stars", "high_rating", "is_open", "review_count",
              "cuisine", "categories"]
other_cols = [c for c in restaurants.columns if c not in first_cols]
restaurants = restaurants[first_cols + other_cols]

restaurants.to_csv(OUTPUT_PATH, index=False)
print(f"Saved {len(restaurants)} restaurants and {restaurants.shape[1]} columns to {OUTPUT_PATH}")
restaurants.head()

Saved 52263 restaurants and 65 columns to /Users/tabeshakbari/Desktop/machine learning project/data/restaurants_clean_v2.csv


,business_id,name,address,city,state,postal_code,metro,in_us,latitude,longitude,stars,high_rating,is_open,review_count,cuisine,categories,price_level,noise_level,outdoor_seating,delivery,takeout,alcohol,reservations,good_for_groups,good_for_kids,has_tv,wifi,attire,bike_parking,caters,table_service,wheelchair_accessible,happy_hour,dogs_allowed,accepts_credit_cards,drive_thru,noise_level_num,ambience_casual,ambience_classy,ambience_divey,ambience_hipster,ambience_intimate,ambience_romantic,ambience_touristy,ambience_trendy,ambience_upscale,parking_garage,parking_street,parking_lot,parking_valet,parking_validated,meal_breakfast,meal_brunch,meal_lunch,meal_dinner,meal_dessert,meal_latenight,parking_any,num_categories,is_fast_food,is_bar,is_chain,days_open,open_weekend,open_late
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,Philadelphia,1,39.955505,-75.155564,4.0,1,1,80,Cafe & Bakery,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...",1.0,NaN,0.0,0.0,1.0,none,NaN,NaN,NaN,NaN,free,NaN,1.0,1.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,5,0,0,0,7.0,1.0,0.0
5,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,Nashville,1,36.269593,-87.058943,2.0,0,1,6,Burgers,"Burgers, Fast Food, Sandwiches, Food, Ice Crea...",1.0,NaN,1.0,1.0,1.0,none,0.0,1.0,1.0,1.0,no,casual,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,1,0,1,7.0,1.0,1.0
8,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,St. Louis,1,38.565165,-90.321087,3.0,0,0,19,Italian,"Pubs, Restaurants, Italian, Bars, American (Tr...",1.0,average,1.0,0.0,1.0,full_bar,0.0,1.0,1.0,1.0,free,casual,NaN,1.0,NaN,NaN,NaN,NaN,1.0,NaN,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,7,0,1,0,NaN,NaN,NaN
9,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,Nashville,1,36.208102,-86.768170,1.5,0,1,10,Burgers,"Ice Cream & Frozen Yogurt, Fast Food, Burgers,...",1.0,NaN,1.0,1.0,1.0,none,0.0,0.0,1.0,1.0,no,casual,NaN,0.0,0.0,1.0,0.0,0.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,0.0,5,1,0,1,7.0,1.0,0.0
11,eEOYSgkmpB90uNA7lDOMRA,Vietnamese Food Truck,,Tampa Bay,FL,33602,Tampa,1,27.955269,-82.456320,4.0,1,1,10,Vietnamese,"Vietnamese, Food, Restaurants, Food Trucks",NaN,NaN,NaN,NaN,NaN,none,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0,0,0,7.0,1.0,0.0


## Column guide for `restaurants_clean.csv`

| Column(s) | What it means |
|---|---|
| `business_id` | Unique Yelp ID (use this to join with other Yelp files) |
| `name`, `address`, `city`, `state`, `postal_code` | Where the restaurant is (city names and ZIPs cleaned) |
| `metro` | Which of the 11 Yelp metro areas it belongs to |
| `in_us` | 1 = U.S., 0 = Canada (Edmonton). Census data only works for U.S. |
| `latitude`, `longitude` | Map location (for counting nearby competitors, matching to Census areas) |
| `stars` | Average Yelp rating (1–5). **Don't use as a feature when predicting `high_rating`.** |
| `high_rating` | **Main target:** 1 if 4+ stars |
| `is_open` | 1 = still open, 0 = closed (Research Question 5) |
| `review_count` | Number of reviews |
| `cuisine` | Main cuisine group (about 20 groups + Other) |
| `categories` | Original Yelp category text |
| `price_level` | 1–4 ($ to $$$$) |
| `noise_level`, `noise_level_num` | Text (quiet → very_loud) and number (1–4) versions |
| `alcohol`, `wifi`, `attire` | Text categories |
| `outdoor_seating`, `delivery`, `takeout`, `reservations`, ... | Yes/no features: 1 = yes, 0 = no, blank = unknown |
| `ambience_*` | Yes/no for each vibe (casual, classy, romantic, trendy, ...) |
| `parking_*`, `parking_any` | Yes/no for each parking type, and any parking at all |
| `meal_*` | Yes/no for which meals it's good for |
| `num_categories`, `is_fast_food`, `is_bar`, `is_chain` | Extra features made from categories and names |
| `days_open`, `open_weekend`, `open_late` | Features made from opening hours |

## AI usage documentation

*(Our professor requires us to show which parts were done by AI and which were done by us. Fill this in honestly as you go.)*

| Part | Who did it | Notes |
|---|---|---|
| First draft of this whole cleaning notebook (all code and explanations) | Claude (AI) | Asked Claude on Nov 8 to write a cleaning notebook for the Yelp business file with simple explanations |
| Running the notebook and checking the output | *Our team (fill in names)* | |
| Changes we made to the AI's code | *Fill in* | e.g., changed cuisine groups, added/removed features |
| Decisions we made ourselves | *Fill in* | e.g., which attributes to keep, 4-star cutoff (from our proposal) |

In [ ]:
import pandas as pd

# Load the clean file (change the path if yours is different)
restaurants = pd.read_csv(
    "/Users/tabeshakbari/Desktop/machine learning project/data/restaurants_clean.csv",
    dtype={"postal_code": str}
)

# Word(s) to search for in the restaurant name
search = ""

# Find every restaurant whose name contains the search word
# case=False means capital letters don't matter ("Luna" = "luna" = "LUNA")
matches = restaurants[restaurants["name"].str.contains(search, case=False, na=False)]

print("Restaurants found:", len(matches))
matches[["name", "city", "state", "stars"]]   # short list so you can pick one

Restaurants found: 1


,name,city,state,stars
29577,Lure Fish House,Santa Barbara,CA,4.5
